In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("landing_timestamp", "")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
dbutils.widgets.text("table_name", "")
table_name = dbutils.widgets.get("table_name")

In [0]:
dbutils.widgets.text("merge_keys", "")
merge_keys = dbutils.widgets.get("merge_keys")
dbutils.widgets.text("load_type", "")
load_type = dbutils.widgets.get("load_type")

In [0]:
bronze_df=spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

In [0]:
updated_df=bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())

In [0]:
load_type = dbutils.widgets.get("load_type").strip().upper()
merge_keys = dbutils.widgets.get("merge_keys").strip()
target_table = f"adb_caresync.silver.{table_name}"
source_count = updated_df.count()

print("=" * 60)
print("BRONZE -> SILVER LOAD")
print("=" * 60)
print(f"  Table             : {table_name}")
print(f"  Target            : {target_table}")
print(f"  Load type         : {load_type}")
print(f"  Landing timestamp : {landing_timestamp}")
print(f"  Merge keys        : {merge_keys or '-'}")
print(f"  Source rows       : {source_count}")
print("-" * 60)

if load_type == "FULL":
    print("[FULL] Overwriting target table...")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print(f"[FULL] Done. {source_count} rows written.")
elif load_type == "APPEND":
    print("[APPEND] Appending rows to target table...")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"[APPEND] Done. {source_count} rows appended.")
elif load_type == "MERGE":
    if not spark.catalog.tableExists(target_table):
        print("[MERGE] Target table does not exist. Creating it with initial load...")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"[MERGE] Done. Table created with {source_count} rows.")
    else:
        updated_df.createOrReplaceTempView("updated_df")
        merge_key_list = [k.strip() for k in merge_keys.split(",") if k.strip()]
        if not merge_key_list:
            raise ValueError("merge_keys must be provided for MERGE load type")
        merge_condition = " AND ".join(
            f"target.{key} = source.{key}" for key in merge_key_list
        )
        update_set = ", ".join(
            f"target.{column} = source.{column}"
            for column in spark.table(target_table).columns
            if column != "insert_timestamp"
        )
        if update_set:
            matched_clause = f"WHEN MATCHED THEN UPDATE SET {update_set}"
        else:
            matched_clause = ""

        print(f"[MERGE] Merge condition: {merge_condition}")
        print("[MERGE] Running MERGE INTO target table...")
        merge_sql = (
            f"MERGE INTO {target_table} AS target "
            f"USING updated_df AS source "
            f"ON {merge_condition} "
            f"{matched_clause} "
            "WHEN NOT MATCHED THEN INSERT *"
        )
        merge_result = spark.sql(merge_sql).first().asDict()
        print("[MERGE] Done.")
        print(f"  Rows affected : {merge_result.get('num_affected_rows')}")
        print(f"  Rows updated  : {merge_result.get('num_updated_rows')}")
        print(f"  Rows inserted : {merge_result.get('num_inserted_rows')}")
else:
    raise ValueError(f"Invalid load_type {load_type!r}. Expected FULL, APPEND or MERGE")

print("-" * 60)
print(f"  Target row count now : {spark.table(target_table).count()}")
print("=" * 60)

        


In [0]:
record_count=updated_df.count()
dbutils.notebook.exit(str(record_count))